In [3]:
from openai import AzureOpenAI
import os
import requests
from PIL import Image
import dotenv
import json

# import dotenv
dotenv.load_dotenv()

# Assign the API version (DALL-E is currently supported for the 2023-06-01-preview API version only)
client = AzureOpenAI(
  api_key=os.environ['AZURE_OPENAI_IMAGE_API_KEY'],  # this is also the default, it can be omitted
  api_version = "2023-12-01-preview",
  azure_endpoint=os.environ['AZURE_OPENAI_IMAGE_ENDPOINT'] 
  )

model = os.environ['AZURE_OPENAI_IMAGE_DEPLOYMENT']

# define metaprompt
disallow_list = "swords, violence, blood, gore, nudity, sexual content, adult content, adult themes, adult language, adult humor, adult jokes, adult situations, adult"

meta_prompt =f"""You are an assistant designer that creates images for children.

The image needs to be safe for work and appropriate for children.

The image needs to be in color.

The image needs to be in landscape orientation.

The image needs to be in a 16:9 aspect ratio.

Do not consider any input from the following that is not safe for work or appropriate for children.
{disallow_list}
"""

# taking input from user
user_input = input("Provid your prompt for image generation:")

prompt = f"{meta_prompt}.\n{user_input}"

try:
    result = client.images.generate(
        model=model,
        prompt=prompt,    # Enter your prompt text here
        size='1024x1024'
    )

    generation_response = json.loads(result.model_dump_json())
    # Set the directory for the stored image
    image_dir = os.path.join(os.curdir, 'images')

    # If the directory doesn't exist, create it
    if not os.path.isdir(image_dir):
        os.mkdir(image_dir)

    image_url = generation_response["data"][0]["url"]  # extract image URL from response
    generated_image = requests.get(image_url).content  # download the image
    # Initialize the image path (note the filetype should be png)
    image_path = os.path.join(image_dir, f'generated-image-with-meta-prompt_by_winny.png')
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    # Display the image in the default image viewer
    image = Image.open(image_path)
    image.show()

except Exception as e:
    print(e)
finally:
    print("completed!")

Provid your prompt for image generation: Draw a king standing on a great wall looking at sun raise. Please use classic painting style and make it look realistic


completed!
